In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import random
import re
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
train_file = '/kaggle/input/nlp-getting-started/train.csv'
test_file = '/kaggle/input/nlp-getting-started/test.csv'
sample_file = '/kaggle/input/nlp-getting-started/sample_submission.csv'

In [ ]:
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
sample_df = pd.read_csv(sample_file)

# EDA

In [ ]:
train_df.head()

In [ ]:
train_df = train_df.drop(columns = ['keyword', 'location'], axis =1 )

In [ ]:
train_df.isna().sum()

In [ ]:
train_df.info()

In [ ]:
train_df['target'].value_counts()

In [ ]:
train_df['text'].str.len().hist();

In [ ]:
for _ in range(50):
    random_row = random.randint(0, len(train_df))
    print("------------------------")
    print(train_df.iloc[random_row]["text"], train_df.iloc[random_row]["target"])

# Preparing Data

## Text Cleaning

In [ ]:
def normalize_repeating(text):
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)        # normalize repeated chars
    text = re.sub(r"([!?.,])\1{1,}", r"\1", text)     # normalize repeated punct
    return text

def fix_urls(text):
    text = re.sub(r"http\S+", "<URL>", text)          # replace URLs
    return text

def fix_mentions(text):
    text = re.sub(r"@\w+", "<USER>", text)            # replace mentions
    return text

def fix_hastag(text):
    text = re.sub(r"#(\w+)", r"\1", text)             # remove "#" but keep word
    return text

def text_cleaning(text):
    text = normalize_repeating(text)
    text = fix_urls(text)
    text = fix_mentions(text)
    text = fix_hastag(text)
    return text

In [ ]:
train_df['text'] = train_df['text'].apply(text_cleaning)

In [ ]:
train_df = train_df.rename(columns={'target': 'labels'})

## Preparing Data

In [ ]:
dataset = Dataset.from_pandas(train_df)
dataset

In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

In [ ]:
train_val = dataset["train"].train_test_split(test_size=0.2, seed=42)

In [ ]:
train_dataset = train_val["train"]
val_dataset = train_val["test"]
test_dataset = dataset["test"]

In [ ]:
print (f'Train len: {len(train_dataset)}, Test len: {len(val_dataset)}, Valid len: {len(test_dataset)}')

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

In [ ]:
train_dataset = train_dataset.remove_columns(["text", "id"]) 
val_dataset = val_dataset.remove_columns(["text", "id"]) 
test_dataset = test_dataset.remove_columns(["text", "id"]) 

In [ ]:
print(train_dataset[0])

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "eval_f1": f1_score(labels, predictions, average="weighted")}

# Model Training

In [ ]:
print("Train size:", len(train_dataset))
print("Eval size:", len(val_dataset))

In [ ]:
num_labels=2 # since targets are 0, 1.
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    logging_dir="./logs",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# Conclusion

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

In [ ]:
predictions = trainer.predict(test_dataset)
y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Test Set")
plt.show()

In [ ]:
print(classification_report(y_true, y_pred))

## Submission

In [ ]:
test_df = test_df.drop(columns=["keyword", "location"], axis=1)
test_df["text"] = test_df["text"].apply(text_cleaning)

In [ ]:
test_dataset = Dataset.from_pandas(test_df)
test_dataset

In [ ]:
test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns(["text", "id"])

In [ ]:
y_pred = trainer.predict(test_dataset)
test_df["target"] = y_pred.predictions.argmax(axis=1)

In [ ]:
test_df.drop(columns=["text"], axis=1).to_csv("submission.csv", index=False)